# CNNs in PyTorch — Hands-On (2 hours)

**Course:** ML, Deep Learning & Computer Vision — Phase 4  
**Prerequisites:** CNN Theory notebook, PyTorch Deep Dive  

---

| Time | What |
|------|------|
| 0:00–0:20 | PyTorch conv layers, shapes, and feature maps |
| 0:20–0:45 | Build a CNN from scratch for CIFAR-10 |
| 0:45–1:10 | Train and evaluate |
| 1:10–1:30 | Visualise: filters, feature maps, Grad-CAM |
| 1:30–1:50 | Transfer learning with pretrained ResNet |
| 1:50–2:00 | Summary and experiments |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import numpy as np
import matplotlib.pyplot as plt
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch {torch.__version__} | Device: {device}")

---
# 1. PyTorch Conv Layers — Understanding the Shapes

## 1.1 `nn.Conv2d` — the fundamental building block

```python
nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0)
```

- **Input shape:** `(batch, C_in, H, W)`
- **Output shape:** `(batch, C_out, H_out, W_out)`
- **Parameters:** `kernel_size² × C_in × C_out + C_out`

In [ ]:
# === Conv2d shape experiments ===
# Input: batch of 4 RGB images, 32×32
x = torch.randn(4, 3, 32, 32)
print(f"Input: {x.shape}  (batch=4, channels=3, H=32, W=32)")

# Basic conv: 3 channels → 16 channels, 3×3 kernel, same padding
conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
out1 = conv1(x)
print(f"\nConv(3→16, k=3, p=1): {out1.shape}  ← same spatial size")
print(f"  Parameters: {sum(p.numel() for p in conv1.parameters()):,}")
print(f"  = 3×3×3×16 + 16 = {3*3*3*16} + 16 = {3*3*3*16+16}")

# Stride 2: downsample by 2×
conv2 = nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1)
out2 = conv2(x)
print(f"\nConv(3→32, k=3, s=2, p=1): {out2.shape}  ← halved spatial size")

# 1×1 conv: channel mixing only
conv1x1 = nn.Conv2d(16, 8, kernel_size=1)
out1x1 = conv1x1(out1)
print(f"\nConv 1×1 (16→8): {out1x1.shape}  ← channels reduced, spatial unchanged")
print(f"  Parameters: {sum(p.numel() for p in conv1x1.parameters()):,} (very cheap!)")

# Pooling
pool = nn.MaxPool2d(2, 2)
out_pooled = pool(out1)
print(f"\nMaxPool2d(2,2): {out1.shape} → {out_pooled.shape}  ← halved, no params")

# Global average pooling
gap = nn.AdaptiveAvgPool2d(1)
out_gap = gap(out1)
print(f"\nAdaptiveAvgPool2d(1): {out1.shape} → {out_gap.shape}  ← one value per channel")

In [ ]:
# === Key shape rule: trace through a mini CNN ===
print("=== Shape trace through a mini CNN ===")
print(f"{'Layer':30s} {'Output shape':>20s} {'Params':>10s}")
print("-" * 65)

x = torch.randn(1, 3, 32, 32)
layers = [
    ("Input", None),
    ("Conv2d(3, 32, 3, pad=1)", nn.Conv2d(3, 32, 3, padding=1)),
    ("BatchNorm2d(32)", nn.BatchNorm2d(32)),
    ("ReLU", nn.ReLU()),
    ("MaxPool2d(2)", nn.MaxPool2d(2)),
    ("Conv2d(32, 64, 3, pad=1)", nn.Conv2d(32, 64, 3, padding=1)),
    ("BatchNorm2d(64)", nn.BatchNorm2d(64)),
    ("ReLU", nn.ReLU()),
    ("MaxPool2d(2)", nn.MaxPool2d(2)),
    ("Conv2d(64, 128, 3, pad=1)", nn.Conv2d(64, 128, 3, padding=1)),
    ("BatchNorm2d(128)", nn.BatchNorm2d(128)),
    ("ReLU", nn.ReLU()),
    ("AdaptiveAvgPool2d(1)", nn.AdaptiveAvgPool2d(1)),
    ("Flatten", nn.Flatten()),
    ("Linear(128, 10)", nn.Linear(128, 10)),
]

print(f"{'Input':30s} {str(tuple(x.shape)):>20s} {'':>10s}")
for name, layer in layers[1:]:
    x = layer(x)
    p = sum(p.numel() for p in layer.parameters()) if hasattr(layer, 'parameters') else 0
    p_str = f"{p:,}" if p > 0 else "-"
    print(f"{name:30s} {str(tuple(x.shape)):>20s} {p_str:>10s}")

---
# 2. Build a CNN for CIFAR-10

CIFAR-10: 60,000 colour images (32×32), 10 classes (airplane, car, bird, cat, deer, dog, frog, horse, ship, truck).

## 2.1 Data pipeline

In [ ]:
# Data augmentation (training only) + normalisation
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),       # 50% chance to flip
    transforms.RandomCrop(32, padding=4),     # random crop with padding
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),   # CIFAR-10 mean
                         (0.2470, 0.2435, 0.2616)),   # CIFAR-10 std
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616)),
])

train_dataset = datasets.CIFAR10('./data', train=True, download=True, transform=train_transform)
test_dataset = datasets.CIFAR10('./data', train=False, download=True, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

CLASSES = ('airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

print(f"Train: {len(train_dataset):,} | Test: {len(test_dataset):,} | Classes: {len(CLASSES)}")

In [ ]:
# Visualise some samples (unnormalise for display)
def unnormalize(img):
    mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
    std = torch.tensor([0.2470, 0.2435, 0.2616]).view(3, 1, 1)
    return (img * std + mean).clamp(0, 1)

fig, axes = plt.subplots(2, 10, figsize=(16, 3.5))
fig.suptitle('CIFAR-10 samples', fontsize=14, fontweight='bold', y=1.02)
for i in range(20):
    img, label = test_dataset[i]
    ax = axes[i // 10, i % 10]
    ax.imshow(unnormalize(img).permute(1, 2, 0).numpy())
    ax.set_title(CLASSES[label], fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 2.2 The model — a clean CNN from scratch

In [ ]:
class CIFAR10CNN(nn.Module):
    """
    A clean CNN for CIFAR-10.
    Architecture:
      Conv block 1: 3 → 32, pool → 16×16
      Conv block 2: 32 → 64, pool → 8×8
      Conv block 3: 64 → 128, pool → 4×4
      Global average pool → 128
      FC → 10 classes
    """
    
    def __init__(self, num_classes=10):
        super().__init__()
        
        # Block 1: 3 → 32 channels
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),    # (B, 32, 32, 32)
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),   # (B, 32, 32, 32)
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                # (B, 32, 16, 16)
            nn.Dropout2d(0.1),
        )
        
        # Block 2: 32 → 64 channels
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1),   # (B, 64, 16, 16)
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),   # (B, 64, 16, 16)
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                # (B, 64, 8, 8)
            nn.Dropout2d(0.2),
        )
        
        # Block 3: 64 → 128 channels
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1),  # (B, 128, 8, 8)
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), # (B, 128, 8, 8)
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                # (B, 128, 4, 4)
            nn.Dropout2d(0.3),
        )
        
        # Classifier head
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),           # (B, 128, 1, 1)
            nn.Flatten(),                       # (B, 128)
            nn.Linear(128, num_classes),         # (B, 10)
        )
    
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.classifier(x)
        return x

model = CIFAR10CNN().to(device)
print(model)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

# Verify shapes
with torch.no_grad():
    test_out = model(torch.randn(2, 3, 32, 32).to(device))
    print(f"Test forward: (2, 3, 32, 32) → {tuple(test_out.shape)}")

---
# 3. Train the CNN

In [ ]:
# Training utilities (same pattern as Phase 3)
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
        total += len(y)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        loss = criterion(logits, y)
        total_loss += loss.item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
        total += len(y)
    return total_loss / total, correct / total

In [ ]:
# === TRAIN ===
N_EPOCHS = 25
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0

print(f"{'Epoch':>5} {'Train Loss':>12} {'Train Acc':>10} {'Val Loss':>10} {'Val Acc':>9} {'LR':>10}")
print("-" * 60)

t_start = time.time()
for epoch in range(1, N_EPOCHS + 1):
    tl, ta = train_epoch(model, train_loader, criterion, optimizer)
    vl, va = evaluate(model, test_loader, criterion)
    scheduler.step()
    
    history['train_loss'].append(tl)
    history['val_loss'].append(vl)
    history['train_acc'].append(ta)
    history['val_acc'].append(va)
    
    if va > best_val_acc:
        best_val_acc = va
        torch.save(model.state_dict(), 'best_cifar_cnn.pth')
    
    lr = optimizer.param_groups[0]['lr']
    if epoch % 5 == 0 or epoch == 1:
        print(f"{epoch:5d} {tl:12.4f} {ta:10.2%} {vl:10.4f} {va:9.2%} {lr:10.6f}")

elapsed = time.time() - t_start
print(f"\nDone in {elapsed:.0f}s | Best val accuracy: {best_val_acc:.2%}")

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
epochs = range(1, N_EPOCHS + 1)

axes[0].plot(epochs, history['train_loss'], 'o-', color='#378ADD', linewidth=2, markersize=3, label='Train')
axes[0].plot(epochs, history['val_loss'], 's--', color='#D85A30', linewidth=2, markersize=3, label='Val')
axes[0].set_title('Loss', fontweight='bold'); axes[0].legend(); axes[0].grid(True, alpha=0.2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-entropy')

axes[1].plot(epochs, history['train_acc'], 'o-', color='#378ADD', linewidth=2, markersize=3, label='Train')
axes[1].plot(epochs, history['val_acc'], 's--', color='#D85A30', linewidth=2, markersize=3, label='Val')
axes[1].axhline(best_val_acc, color='#1D9E75', linestyle=':', label=f'Best={best_val_acc:.1%}')
axes[1].set_title('Accuracy', fontweight='bold'); axes[1].legend(); axes[1].grid(True, alpha=0.2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')

plt.suptitle('CIFAR-10 CNN Training', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

---
# 4. Visualise What the CNN Learned

## 4.1 First-layer filters

In [ ]:
# Extract and visualise the first conv layer's filters
model.load_state_dict(torch.load('best_cifar_cnn.pth', weights_only=True))
model.eval()

filters = model.block1[0].weight.data.cpu()  # (32, 3, 3, 3)

# Normalise each filter to [0, 1] for display
def normalise_filter(f):
    f = f - f.min()
    f = f / (f.max() + 1e-8)
    return f

fig, axes = plt.subplots(4, 8, figsize=(14, 7))
fig.suptitle('First conv layer filters (3×3 RGB patterns the network learned)',
             fontsize=13, fontweight='bold', y=1.01)

for i, ax in enumerate(axes.flat):
    f = normalise_filter(filters[i].permute(1, 2, 0))  # (3, 3, 3) → (3, 3, 3) HWC
    ax.imshow(f.numpy(), interpolation='nearest')
    ax.set_title(f'F{i}', fontsize=7)
    ax.axis('off')

plt.tight_layout()
plt.show()

print("These are the 32 learned 3×3 RGB filters.")
print("Some detect edges, others respond to specific colours or gradients.")

## 4.2 Feature maps — what the network sees at each layer

In [ ]:
# Pass one image through and visualise feature maps at each block
test_img, test_label = test_dataset[7]
x = test_img.unsqueeze(0).to(device)

# Forward through each block
with torch.no_grad():
    fm1 = model.block1[:4](x)   # after first 2 convs + BN + ReLU (before pool)
    fm2 = model.block2[:4](model.block1(x))
    fm3 = model.block3[:4](model.block2(model.block1(x)))

fig, axes_all = plt.subplots(4, 8, figsize=(14, 7))

# Original image
for ax in axes_all[0]:
    ax.axis('off')
axes_all[0, 0].imshow(unnormalize(test_img).permute(1, 2, 0).numpy())
axes_all[0, 0].set_title(f'Input: {CLASSES[test_label]}', fontweight='bold', fontsize=10)

# Feature maps from each block
for row, (fm, name) in enumerate([(fm1, 'Block 1'), (fm2, 'Block 2'), (fm3, 'Block 3')], 1):
    for col in range(8):
        if col < fm.shape[1]:
            axes_all[row, col].imshow(fm[0, col].cpu().numpy(), cmap='viridis')
        axes_all[row, col].axis('off')
    axes_all[row, 0].set_ylabel(name, fontsize=10, rotation=0, labelpad=40, fontweight='bold')

plt.suptitle('Feature maps at each depth (first 8 channels)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f"Block 1 features: {tuple(fm1.shape)} — edges and textures")
print(f"Block 2 features: {tuple(fm2.shape)} — parts and shapes")
print(f"Block 3 features: {tuple(fm3.shape)} — object-level patterns")

## 4.3 Per-class accuracy and confusion matrix

In [ ]:
# Per-class evaluation
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for X, y in test_loader:
        logits = model(X.to(device))
        all_preds.append(logits.argmax(1).cpu())
        all_labels.append(y)

all_preds = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

# Confusion matrix
cm = np.zeros((10, 10), dtype=int)
for t, p in zip(all_labels, all_preds):
    cm[t, p] += 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks(range(10)); axes[0].set_yticks(range(10))
axes[0].set_xticklabels(CLASSES, rotation=45, ha='right', fontsize=8)
axes[0].set_yticklabels(CLASSES, fontsize=8)
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
axes[0].set_title('Confusion matrix', fontweight='bold')
plt.colorbar(im, ax=axes[0], shrink=0.8)

# Per-class accuracy
class_acc = cm.diagonal() / cm.sum(axis=1)
colors = ['#1D9E75' if a > 0.85 else '#D85A30' if a < 0.7 else '#378ADD' for a in class_acc]
axes[1].barh(CLASSES, class_acc, color=colors)
axes[1].set_xlim(0, 1)
axes[1].set_title('Per-class accuracy', fontweight='bold')
axes[1].set_xlabel('Accuracy')
for i, a in enumerate(class_acc):
    axes[1].text(a + 0.01, i, f'{a:.1%}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nOverall accuracy: {(all_preds == all_labels).mean():.2%}")
print(f"Hardest class:    {CLASSES[class_acc.argmin()]} ({class_acc.min():.1%})")
print(f"Easiest class:    {CLASSES[class_acc.argmax()]} ({class_acc.max():.1%})")

---
# 5. Transfer Learning with Pretrained ResNet

Instead of training from scratch, we can use a ResNet pretrained on ImageNet (1.2M images, 1000 classes) and **fine-tune** it for CIFAR-10.

**Strategy:**
1. Load pretrained ResNet-18
2. Replace the final FC layer (1000 → 10 classes)
3. Optionally freeze early layers (they already detect general features)
4. Train with a smaller learning rate

In [ ]:
# Load pretrained ResNet-18
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Replace the final FC layer
num_features = resnet.fc.in_features  # 512
resnet.fc = nn.Linear(num_features, 10)

resnet = resnet.to(device)

total_params = sum(p.numel() for p in resnet.parameters())
trainable = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
print(f"ResNet-18 for CIFAR-10:")
print(f"  Total params:     {total_params:,}")
print(f"  Trainable params: {trainable:,}")
print(f"  New FC layer:     512 → 10 = {512*10+10:,} params")

In [ ]:
# Fine-tune with smaller lr
optimizer_ft = torch.optim.Adam(resnet.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler_ft = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=10)

print("Fine-tuning ResNet-18 for 10 epochs...")
ft_history = {'train_acc': [], 'val_acc': []}

for epoch in range(1, 11):
    tl, ta = train_epoch(resnet, train_loader, criterion, optimizer_ft)
    vl, va = evaluate(resnet, test_loader, criterion)
    scheduler_ft.step()
    ft_history['train_acc'].append(ta)
    ft_history['val_acc'].append(va)
    if epoch % 2 == 0:
        print(f"  Epoch {epoch:2d} | Train: {ta:.2%} | Val: {va:.2%}")

print(f"\nBest val accuracy: {max(ft_history['val_acc']):.2%}")
print(f"Compare with our CNN from scratch: {best_val_acc:.2%}")
print(f"\nTransfer learning typically achieves higher accuracy with fewer epochs.")

In [ ]:
# Compare: from scratch vs transfer learning
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, N_EPOCHS+1), history['val_acc'], 'o-', color='#378ADD',
        linewidth=2, markersize=3, label=f'CNN from scratch (best={best_val_acc:.1%})')
ax.plot(range(1, 11), ft_history['val_acc'], 's-', color='#1D9E75',
        linewidth=2, markersize=4, label=f'ResNet-18 fine-tuned (best={max(ft_history["val_acc"]):.1%})')
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation accuracy')
ax.set_title('From scratch vs transfer learning', fontweight='bold')
ax.legend(fontsize=11); ax.grid(True, alpha=0.2)
plt.tight_layout(); plt.show()

print("Transfer learning wins because ImageNet features (edges, textures, shapes)")
print("are universal — they transfer to almost any vision task.")

---
# 6. Experiments to Try

| Experiment | Change | Expected effect |
|-----------|--------|----------------|
| 1. No data augmentation | Remove `RandomHorizontalFlip` and `RandomCrop` | Lower val acc (overfitting) |
| 2. No batch norm | Remove `BatchNorm2d` layers | Slower convergence, might not train |
| 3. Deeper network | Add a 4th conv block (128→256) | Might improve, or overfit on CIFAR |
| 4. No pooling, use stride | Replace `MaxPool2d(2)` with `stride=2` in conv | Similar accuracy, different compute profile |
| 5. Larger kernels | Use 5×5 in first layer | Slightly better first-layer features |
| 6. Freeze ResNet backbone | Set `requires_grad=False` for all but last layer | Faster training, slightly lower acc |
| 7. Different pretrained model | Try `resnet34`, `resnet50`, `efficientnet_b0` | Better models = better features |

## Key takeaways

| Concept | PyTorch | What you learned |
|---------|---------|------------------|
| Conv layer | `nn.Conv2d(C_in, C_out, k, padding=k//2)` | Weight sharing, local connectivity |
| Pooling | `nn.MaxPool2d(2, 2)` | Spatial downsampling, no parameters |
| Batch norm | `nn.BatchNorm2d(C)` | Stabilise training, mild regularisation |
| Dropout2d | `nn.Dropout2d(p)` | Drop entire feature maps, not individual pixels |
| Data augmentation | `transforms.RandomHorizontalFlip()` etc. | Free regularisation from data |
| Global avg pool | `nn.AdaptiveAvgPool2d(1)` | Any input size → fixed-size output |
| Transfer learning | `models.resnet18(weights=...)` | Pretrained features transfer to new tasks |

**Next:** Apply CNNs to your own image classification task (Phase 4 capstone project).

In [ ]:
# Clean up
import os
if os.path.exists('best_cifar_cnn.pth'):
    os.remove('best_cifar_cnn.pth')
    print("Cleaned up.")